# ☀️ **Actividad 1 A2IC — Optimización bajo incertidumbre** — **Solución**
### **EL4114 – Optimización · Departamento de Ingeniería Eléctrica · Universidad de Chile**

## 💻 **Recomendaciones**

* Este notebook se complementa con la guía **Actividad 1 A2IC — Optimización bajo incertidumbre**. Lea las instrucciones en ella para obtener las pautas de modelamiento necesarias para la implementación en este documento."

* En cada pregunta se incluyen bloques en blanco para programar sus soluciones. Esta es solo una recomendación de estructura. Usted puede implementar su código como le sea conveniente.

* Realice una copia de este notebook en su cuenta de Google y resuelva allí. Para realizarla, siga la ruta **"Archivo $\rightarrow$ Guardar una copia en Drive"**

---

## 🛠️ Modelamiento e implementación

En el mundo laboral utilizarán herramientas de IA para ser más eficientes. Por ello, en esta entrega se fomenta su uso estratégico. El objetivo es que deleguen en la IA las tareas repetitivas de formato o sintaxis, mientras ustedes se concentran en el verdadero núcleo del problema: **el modelamiento y el análisis de los resultados.**

⚠️ **Regra principal:** Antes de escribir cualquier línea de código, deben estructurar matemáticamente el problema en papel o en su editor de texto científico. Si decides usar IA para generar un fragmento de código, debes ser capaz de explicar línea por línea qué hace esa función en una eventual defensa o corrección.

---

### ⚖️ Guía para el uso ético

Para el desarrollo de esta actividad, se permite y fomenta el uso de herramientas de IA (como ChatGPT, Claude, Gemini, Copilot, etc.) como **asistentes de aprendizaje y co-programadores**, bajo las siguientes pautas:

### 🟢 Usos Recomendados

> 📝 **Sintaxis:** Si tu formulación en papel es correcta, pero olvidas cómo declarar variables indexadas por múltiples conjuntos en tu librería de optimización, pídele a la IA ejemplos genéricos de sintaxis.

> 🔍 **Errores:** Si el optimizador arroja un error de código o de factibilidad (*Infeasible*), copia el error en la IA y pídele que te ayude a interpretar qué restricción o lógica matemática podría estar fallando.

> 🧼 **Limpieza de Código:** Utiliza la IA para mejorar la legibilidad de tus gráficos o para estructurar de manera más limpia tus bucles e impresiones en consola.

---

### 🔴 Prácticas no permitidas

> 📦 **Copiado a Ciegas:** Evita pedirle a la IA que escriba el modelo completo desde cero. El código generado por IA suele omitir restricciones sutiles o inventar sintaxis que no corresponde a la librería, lo que te costará más tiempo depurar que escribirlo tú mismo.

> 🧠 **Delegar el Pensamiento Crítico:** La IA no debe responder las preguntas de discusión por ti. Las herramientas de IA suelen redactar análisis genéricos y superficiales. Tu evaluación debe fundamentarse estrictamente en los datos, números y gráficos que arrojó **tu** simulación.

# **Instalación de dependencias**
Ejecute esta celda una sola vez (en Colab tarda unos segundos).

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"   # cambie a "notebook" si usa Jupyter local, o "browser"

## **Instalación de Gurobi**
Antes de resolver el problema revise el tutorial de gurobi disponible en [link](https://drive.google.com/file/d/1n3iC3TfnXAqgOkLUMMeV3gVTcnegJAvN/view?usp=sharing)

In [2]:
# Subir la licencia a colab (archivo gurobi.lic)
#from google.colab import files
#uploaded = files.upload()

#!pip install gurobipy
import gurobipy as grb

In [3]:
# Se crea una variable de entorno para luego acceder a la licencia
#import os
#os.environ['GRB_LICENSE_FILE'] = '/content/gurobi.lic'
#print(os.environ['GRB_LICENSE_FILE'])

# Se verifica la versión de Gurobi que se está usando
#print(grb.gurobi.version())

# Sección 1 — Modelo determinista con reservas


## 1.1 Datos del sistema y visualización del pronóstico base

In [4]:
# ---------------- DATOS DEL SISTEMA ----------------
T = list(range(24))                 # periodos 0..23 (horas)
G = ['G1', 'G2', 'G3']              # unidades convencionales

Pmin   = {'G1': 40,  'G2': 20,  'G3': 10}    # MW
Pmax   = {'G1': 150, 'G2': 100, 'G3': 80}    # MW
cvar   = {'G1': 20,  'G2': 40,  'G3': 70}    # $/MWh  (G1 base barata, G3 peaker cara)
cnl    = {'G1': 100, 'G2': 80,  'G3': 50}    # $/h    costo de estar encendida
cstart = {'G1': 500, 'G2': 300, 'G3': 100}   # $      costo de partida
VOLL   = 10000                               # $/MWh  costo de energia no suministrada

# Perfil horario de demanda (normalizado 0-1) y escala a MW
perfil_D = np.array([0.65,0.60,0.58,0.56,0.58,0.62,0.70,0.80,0.88,0.92,0.95,0.97,
                     0.98,0.96,0.93,0.92,0.94,1.00,0.98,0.95,0.90,0.85,0.78,0.70])
D_base = 248 * perfil_D                       # MW

# Perfil solar: campana diurna (cero de noche, maximo ~mediodia)
Psol_base = np.array([max(0.0, 150*np.sin(np.pi*(h-6)/12)) if 6 <= h <= 18 else 0.0
                      for h in T])

print("Capacidad convencional total:", sum(Pmax.values()), "MW")
print("Demanda maxima (base):", round(D_base.max(),1), "MW   |   Solar maxima (base):", round(Psol_base.max(),1), "MW")

Capacidad convencional total: 330 MW
Demanda maxima (base): 248.0 MW   |   Solar maxima (base): 150.0 MW


In [5]:
# Visualizacion del pronostico determinista (base)
fig = go.Figure()
fig.add_bar(x=T, y=Psol_base, name="Solar disponible", marker_color="#f4a261")
fig.add_scatter(x=T, y=D_base, name="Demanda", mode="lines+markers",
                line=dict(color="#264653", width=3))
fig.add_hline(y=sum(Pmax.values()), line_dash="dash", line_color="crimson",
              annotation_text="Capacidad convencional total")
fig.update_layout(title="Sección 1 — Pronóstico determinista (escenario base)",
                  xaxis_title="Hora", yaxis_title="MW", template="plotly_white",
                  legend=dict(orientation="h"))
fig.show()

## 1.2 Desarrolle una función llamada resolver_uc_reserva() en Gurobi que resuelva el problema de Unit Commitment para distintos niveles de reservas (utilice el enfoque A, $\alpha$ será parámetro de entrada de la función). La función debe retornar los valores óptimos de las variables de decisión (estados de encendido, despachos de potencia y energía no suministrada).

## 1.3 Ejecute el modelo de forma iterativa para evaluar cuatro niveles de margen de reserva: $R \in \{0\%,\,10\%,\,15\%,\,20\%\}$. El caso con $0\%$ de reserva actuará como la base determinista pura sin criterios de seguridad.



## 1.4 Realice las siguientes gráficas:
1) Un diagrama de área apilada para el caso sin reserva (0%) y el caso más conservador (20%) que muestre el aporte de cada unidad térmica, la generación solar utilizada y la Energía No Suministrada (ENS) frente a la curva de demanda.
2) Una comparación visual ( del estado de encendido/apagado de las unidades entre el escenario sin reserva (0%) y el escenario conservador (20%).
3) Un gráfico de barras que compare el costo total de operación del sistema para cada uno de los cuatro márgenes de reserva evaluados.

# Sección 2 -  Modelo estocástico mediante SAA

## 1.1 Generación de escenarios
En vez de usar un único pronóstico con reservas conservadoras, modelamos la incertidumbre con **distribuciones de probabilidad** y aplicamos **Sample Average Approximation (SAA)**:
- Factor de demanda: $k_D \sim \mathcal{N}(1.0,\; 0.20^2)$
- Factor solar: $k_S \sim \mathcal{N}(1.0,\; 0.50^2)$ (mayor incertidumbre en energía solar)
- Correlación negativa: $\rho = -0.5$ (día nublado → menos solar, más demanda)

El método SAA consiste en:
1. Muestrear $N$ realizaciones $(k_D^{(i)}, k_S^{(i)})$ desde la distribución conjunta
2. Construir escenarios: $D_t^{(i)} = k_D^{(i)} \cdot D_t^{\text{base}}$, $\;P_t^{sol,(i)} = k_S^{(i)} \cdot P_t^{sol,\text{base}}$
3. Resolver el UC estocástico con estos $N$ escenarios equiprobables ($\pi_i = 1/N$)


In [6]:
def generar_escenarios_SAA(N, seed=42):
    """Muestrea N escenarios de (kD, kS) desde distribucion normal bivariada."""
    rng = np.random.default_rng(seed)
    mu = [1.0, 1.0]
    sigma_D, sigma_S, rho = 0.1, 0.3, -0.5
    cov = [[sigma_D**2, rho*sigma_D*sigma_S],
           [rho*sigma_D*sigma_S, sigma_S**2]]
    samples = rng.multivariate_normal(mu, cov, size=N)
    samples[:, 0] = np.clip(samples[:, 0], 0.5, 1.6)   # limites fisicos demanda
    samples[:, 1] = np.clip(samples[:, 1], 0.0, 2.0)   # limites fisicos solar

    D_saa, Sol_saa, pi_saa = {}, {}, {}
    for i in range(N):
        s = f"saa_{i}"
        D_saa[s] = samples[i, 0] * D_base
        Sol_saa[s] = samples[i, 1] * Psol_base
        pi_saa[s] = 1.0 / N
    return D_saa, Sol_saa, pi_saa, samples

In [7]:
# Visualizacion de escenarios muestreados (N=100)
_, _, _, muestras_100 = generar_escenarios_SAA(100, seed=42)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Muestras en espacio (kD, kS)",
                                    "Perfiles de demanda muestreados"])

# Scatter de factores
fig.add_trace(go.Scatter(x=muestras_100[:, 0], y=muestras_100[:, 1], mode="markers",
                         marker=dict(size=6, color="#2a9d8f", opacity=0.7),
                         name="Muestras"), row=1, col=1)
fig.add_trace(go.Scatter(x=[1.0], y=[1.0], mode="markers",
                         marker=dict(size=12, color="red", symbol="x"),
                         name="Base (1,1)"), row=1, col=1)

# Spaghetti de perfiles de demanda
D_100, _, _, _ = generar_escenarios_SAA(100, seed=42)
for i, s in enumerate(D_100):
    fig.add_trace(go.Scatter(x=T, y=D_100[s], mode="lines",
                             line=dict(color="#264653", width=0.5), opacity=0.2,
                             showlegend=(i == 0), name="Escenarios",
                             legendgroup="esc"), row=1, col=2)
fig.add_trace(go.Scatter(x=T, y=D_base, mode="lines",
                         line=dict(color="red", width=2.5, dash="dash"),
                         name="Base"), row=1, col=2)

fig.update_xaxes(title_text="kD (factor demanda)", row=1, col=1)
fig.update_yaxes(title_text="kS (factor solar)", row=1, col=1)
fig.update_xaxes(title_text="Hora", row=1, col=2)
fig.update_yaxes(title_text="MW", row=1, col=2)
fig.update_layout(title="Sección 2 — Escenarios muestreados desde distribución bivariada (N=100)",
                  template="plotly_white", height=420, legend=dict(orientation="h"))
fig.show()

## 1.2 Implemente una función llamada resolver_uc() que resuelva el problema de Unit Commitment estocástico de dos etapas. El código debe estructurarse de modo que las decisiones de compromiso (encendido, apagado y arranque de unidades) se consideren variables de primera etapa (únicas para todo el problema), mientras que los niveles de despacho por unidad, la energía solar aprovechada y la energía no suministrada actúen como variables de segunda etapa (específicas para cada escenario generado).

## 1.3 Ejecutar el algoritmo iterativamente evaluando los tamaños de muestra: 2, 5, 10, 25, 50 y 100 escenarios equiprobables. Para cada caso, reporten el costo de la función objetivo obtenido dentro de la muestra.

## 1.4 Contraste el esquema de encendido de las unidades térmicas obtenido con una muestra pequeña ($N = 2$) frente al obtenido con la muestra más grande ($N = 100$).